# Create the Vector Search Endpoint and Index

### Enable Change Data Feed on the source table

In [0]:
%sql
ALTER TABLE policyiq.silver.policy_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

### Create the Vector Search endpoint

In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
%python
from databricks.ai_search.client import VectorSearchClient

vsc = VectorSearchClient()

ENDPOINT_NAME = "policyiq_vs_endpoint"
INDEX_NAME = "policyiq.silver.policy_chunks_index"
SOURCE_TABLE = "policyiq.silver.policy_chunks"

index = vsc.create_delta_sync_index_and_wait(
    endpoint_name=ENDPOINT_NAME,
    index_name=INDEX_NAME,
    source_table_name=SOURCE_TABLE,
    pipeline_type="TRIGGERED",
    primary_key="policy_chunk_id",
    embedding_source_column="chunk_embed_text",
    embedding_model_endpoint_name="databricks-gte-large-en",
    columns_to_sync=[
        "policy_chunk_id", "policy_id", "policy_name", "policy_domain",
        "issuing_authority", "version", "file_name", "chunk_position",
        "chunk_text", "chunk_char_length"
    ]
)

print("Index created:", INDEX_NAME)

In [0]:
%python
index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

results = index.similarity_search(
    query_text="What is the maximum leave approval turnaround time?",
    columns=["policy_id", "chunk_position", "chunk_text"],
    num_results=3
)

for r in results["result"]["data_array"]:
    print(r[0], "| chunk", r[1])
    print(r[2][:300], "...\n")

In [0]:
%python
from databricks.ai_search.client import VectorSearchClient

vsc = VectorSearchClient()
index = vsc.get_index(endpoint_name="policyiq_vs_endpoint", index_name="policyiq.silver.policy_chunks_index")

print(index.describe())